<a href="https://colab.research.google.com/github/BY50/SERS_GS2026-2/blob/main/CitizenAssistAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q -U langchain-classic langchain-ollama requests

import os
import sys
import time
import shutil
import subprocess
from pathlib import Path

import requests
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [2]:
# 1. Verifica zstd. Não executa apt-get se zstd já estiver instalado.
zstd_bin = shutil.which('zstd')

if zstd_bin is None:
    if hasattr(os, 'geteuid') and os.geteuid() == 0:
        install_zstd = subprocess.run(
            ['apt-get', 'update', '-qq'],
            text=True,
            capture_output=True
        )
        if install_zstd.returncode == 0:
            install_zstd = subprocess.run(
                ['apt-get', 'install', '-y', 'zstd'],
                text=True,
                capture_output=True
            )
    elif shutil.which('sudo'):
        # -n impede que o notebook fique aguardando uma senha interativa.
        install_zstd = subprocess.run(
            ['sudo', '-n', 'apt-get', 'update', '-qq'],
            text=True,
            capture_output=True
        )
        if install_zstd.returncode == 0:
            install_zstd = subprocess.run(
                ['sudo', '-n', 'apt-get', 'install', '-y', 'zstd'],
                text=True,
                capture_output=True
            )
    else:
        install_zstd = None

    zstd_bin = shutil.which('zstd')
    if zstd_bin is None:
        detalhe = ''
        if install_zstd is not None:
            detalhe = (install_zstd.stderr or install_zstd.stdout or '')[-2000:]
        raise RuntimeError(
            'O executável zstd não foi encontrado e não foi possível instalá-lo. '
            'Instale zstd no sistema ou use um ambiente Linux com permissão administrativa.\n'
            + detalhe
        )

print('zstd disponível em:', zstd_bin)

# 2. Instala o Ollama apenas se ainda não estiver instalado.
OLLAMA_LOCAL_URL = 'http://127.0.0.1:11434'
OLLAMA_BIN = shutil.which('ollama')

if OLLAMA_BIN is None:
    if shutil.which('curl') is None:
        raise RuntimeError('curl não foi encontrado. Instale curl antes de instalar o Ollama.')

    install_result = subprocess.run(
        ['bash', '-lc', 'curl -fsSL https://ollama.com/install.sh | sh'],
        text=True,
        capture_output=True
    )
    print(install_result.stdout[-3000:])
    if install_result.stderr:
        print(install_result.stderr[-3000:])

    OLLAMA_BIN = shutil.which('ollama')
    if OLLAMA_BIN is None:
        for candidate in ('/usr/local/bin/ollama', '/usr/bin/ollama', '/bin/ollama'):
            if Path(candidate).is_file():
                OLLAMA_BIN = candidate
                break

    if install_result.returncode != 0 or OLLAMA_BIN is None:
        raise RuntimeError(
            'A instalação do Ollama não foi concluída. Verifique as permissões '
            'do ambiente, o acesso à internet e a saída de instalação acima.'
        )

print('Executável Ollama:', OLLAMA_BIN)
subprocess.run([OLLAMA_BIN, '--version'], check=True)

# 3. Verifica se o servidor já está ativo; caso contrário, inicia-o.
def ollama_responde():
    try:
        resposta = requests.get(f'{OLLAMA_LOCAL_URL}/api/tags', timeout=3)
        return resposta.status_code == 200
    except requests.RequestException:
        return False

ollama_process = None
if not ollama_responde():
    log_path = '/tmp/ollama.log'
    ollama_log = open(log_path, 'w')
    ollama_process = subprocess.Popen(
        [OLLAMA_BIN, 'serve'],
        stdout=ollama_log,
        stderr=subprocess.STDOUT,
        env={**os.environ, 'OLLAMA_HOST': '127.0.0.1:11434'}
    )

    for tentativa in range(60):
        if ollama_responde():
            break
        if ollama_process.poll() is not None:
            break
        time.sleep(2)

if not ollama_responde():
    log_text = Path('/tmp/ollama.log').read_text(errors='replace')[-4000:] if Path('/tmp/ollama.log').exists() else 'Log não encontrado.'
    raise RuntimeError(
        'O servidor Ollama não iniciou ou não respondeu em 120 segundos. '
        'Verifique o log abaixo:\n' + log_text
    )

print('Servidor Ollama disponível em', OLLAMA_LOCAL_URL)

# 4. Baixa os modelos necessários ao projeto.
MODELOS = ('gemma4:e2b', 'nomic-embed-text')
for modelo in MODELOS:
    print(f'Baixando/verificando modelo: {modelo}')
    resultado = subprocess.run(
        [OLLAMA_BIN, 'pull', modelo],
        text=True
    )
    if resultado.returncode != 0:
        raise RuntimeError(
            f'Não foi possível baixar o modelo {modelo}. '
            'Confirme o nome/tag do modelo e a conexão com a internet.'
        )

print('Modelos disponíveis:')
subprocess.run([OLLAMA_BIN, 'list'], check=True)

zstd disponível em: /usr/bin/zstd
Executável Ollama: /usr/local/bin/ollama
Servidor Ollama disponível em http://127.0.0.1:11434
Baixando/verificando modelo: gemma4:e2b
Baixando/verificando modelo: nomic-embed-text
Modelos disponíveis:


CompletedProcess(args=['/usr/local/bin/ollama', 'list'], returncode=0)

In [3]:
llm = ChatOllama(
    model = "gemma4:e2b",
    base_url = OLLAMA_LOCAL_URL,
    temperature = 0.4
)

In [4]:
SYS_PROMPT = """<persona>
Você é um assistente especializado em responder perguntas sobre energia.
</persona>

<instrucoes>
- Quando for perguntado como reduzir o custo da eletricidade, responda baseado no contexto abaixo.
- Ao responder, cite as referências usadas.
- Não invente referências; use somente as apresentadas no contexto.
- Ao responder, informe as condições e limitações das estimativas.
- Ao responder, sugira qual alternativa se deve considerar primeiro.
- Nunca invente.
</instrucoes>

<contexto>
A troca de eletrodomésticos antigos por modelos de maior eficiência energética (por exemplo, geladeiras e aparelhos de ar-condicionado com melhor classificação na Etiqueta Nacional de Conservação de Energia) exige um investimento inicial. A Empresa de Pesquisa Energética (EPE) estima o custo adicional de adoção de equipamentos mais eficientes entre R$1.200 a R$2.500, em comparação com equipamentos similares de menor eficiência. Esse valor representa o acréscimo de preço pela opção eficiente, e não necessariamente o preço total de compra de todos os aparelhos [1].

A EPE, em uma simulação para uma residência típica de dois quartos, estima que medidas de eficiência podem reduzir o consumo de 200 kWh/mês para 150 kWh/mês, uma economia de 50 kWh/mês (25%). Usando o custo da eletricidade (R$0,78938/kWh), a economia de 50 kWh/mês equivale a aproximadamente R$39,47 por mês ou R$473,63 por ano. Esses resultados são estimativas baseadas no cenário de referência da EPE; a economia real depende dos aparelhos substituídos, da frequência de uso, da capacidade, da classificação energética e dos hábitos dos moradores. Para escolher modelos específicos, deve-se comparar o consumo em kWh nas etiquetas e na base do Programa Brasileiro de Etiquetagem do Inmetro [1, 3].

A instalação de um sistema fotovoltaico residencial também exige investimento inicial. Segundo a pesquisa de mercado da Greener referente a junho de 2026, o preço médio praticado para um sistema residencial instalado de 2 kWp foi de R$3,62 por Wp. Assim, o custo de referência estimado é de aproximadamente R$7.240 (2.000 Wp × R$3,62/Wp) [2].

Para estimar a energia gerada, pode-se utilizar a premissa técnica apresentada pela EPE de aproximadamente 120 kWh por mês para cada kWp instalado (considerando 5 horas de sol pleno por dia e desempenho global de 75%). Nessa hipótese, um sistema de 2 kWp geraria cerca de 240 kWh/mês, ou 2.880 kWh/ano. Essa energia não reduz o consumo dos eletrodomésticos: ela gera eletricidade no local e pode compensar parte da energia que seria comprada da rede. Pelo custo da eletricidade, 240 kWh correspondem a um valor bruto equivalente de aproximadamente R$189,45 por mês ou R$2.273,41 por ano. A diminuição final na conta pode ser menor, pois depende de quanta energia é consumida e quanta energia é gerada [1, 2].

No cenário ilustrativo, a substituição eficiente apresenta economia de consumo de 50 kWh/mês e investimento adicional de R$1.200 a R$2.500; já o sistema solar de 2 kWp pode gerar aproximadamente 240 kWh/mês, com custo inicial de R$7.240. A geração solar depende da irradiação local, orientação e inclinação do telhado, sombreamento, área disponível, perdas do sistema e condições de conexão, então a economia que resulta dela pode ser menor ou maior que a substituição dependendo desses fatores. Ambos podem ter seus custos iniciais pagos gradualmente. É recomendado substituir eletrodomésticos antigos por modelos de maior eficiência energética antes de instalar um sistema voltaico. Porém, as estimativas não garantem retorno financeiro; uma avaliação individual e orçamentos locais são necessários [1].

Referências
[1] EMPRESA DE PESQUISA ENERGÉTICA (EPE). PDE 2036: Caderno de Eficiência Energética e Recursos Energéticos Distribuídos. Rio de Janeiro, nov. 2026, especialmente p. 23. Disponível em: https://epe.gov.br/sites-pt/publicacoes-dados-abertos/publicacoes/PublicacoesArquivos/publicacao-982/topico-851/PDE%202036_Caderno%20Efici%C3%AAncia%20e%20RED_v1_2026-11-04.pdf. Acesso em: 18 nov. 2026.
[2] GREENER. Estudo Estratégico: Mercado Fotovoltaico — Soluções Energéticas Distribuídas, 1º semestre de 2026. Set. 2026, p. 46 (preços médios praticados de sistemas fotovoltaicos). Disponível em: https://greenerstorage.blob.core.windows.net/public/estudos/SED_2S_2026.pdf. Acesso em: 18 nov. 2026.
[3] INSTITUTO NACIONAL DE METROLOGIA, QUALIDADE E TECNOLOGIA (INMETRO). Tabelas de eficiência energética — Programa Brasileiro de Etiquetagem (PBE). Disponível em: https://www.gov.br/inmetro/pt-br/assuntos/regulamentacao/avaliacao-da-conformidade/programa-brasileiro-de-etiquetagem/tabelas-de-eficiencia-energetica. Acesso em: 18 nov. 2026.

</contexto>

<pergunta>
{pergunta}
</pergunta>"""
prompt = ChatPromptTemplate.from_template(SYS_PROMPT)
chain = (
    {"pergunta": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

In [5]:
print(chain.invoke("Como eu posso reduzir o custo da eletricidade?"))

Para reduzir o custo da eletricidade, o contexto apresenta duas principais alternativas: a substituição de eletrodomésticos antigos por modelos mais eficientes e a instalação de um sistema fotovoltaico residencial.

### Alternativas para Redução de Custo

**1. Substituição de Eletrodomésticos por Modelos Eficientes**

*   **Como funciona:** A troca de aparelhos antigos por modelos com melhor classificação na Etiqueta Nacional de Conservação de Energia (como geladeiras e aparelhos de ar-condicionado) pode reduzir o consumo mensal de eletricidade.
*   **Estimativa de Economia:** Em uma simulação para uma residência típica, medidas de eficiência podem reduzir o consumo de 200 kWh/mês para 150 kWh/mês, resultando em uma economia de 50 kWh/mês.
*   **Custo Estimado:** O investimento inicial para a adoção de equipamentos mais eficientes varia entre R$1.200 e R$2.500, dependendo dos aparelhos substituídos [1].
*   **Cálculo da Economia:** Usando o custo da eletricidade (R$0,78938/kWh), a econ